# 07. Architecture Visualization

## 学習目標
- Model M（Classical GPT, 10M params）の end-to-end データフローとブロック構造を図で把握する
- Classical と Modern の構成差（LayerNorm/RMSNorm, learned/RoPE, GELU/SwiGLU, bias）を対応づける

## Transformer block（residual stream 視点, spec §8.2）
```text
        ┌──────────────── residual stream ────────────────┐
 x ──┬─▶│ Norm → Self-Attention → (+) ─┬─▶ Norm → MLP → (+)│─▶
     └──┘                              └──────────────────┘
Classical: Norm=LayerNorm, pos=learned(加算), MLP=GELU(4d), bias=有
Modern   : Norm=RMSNorm,   pos=RoPE(attn内回転), MLP=SwiGLU(≈8d/3×3), bias=無
```

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"

In [2]:
from jp_llm_lab.models.config import ModelConfig
from jp_llm_lab.models.transformer import ClassicalGPT

cfg_c = ModelConfig(vocab_size=8192, d_model=320, n_heads=8, n_layers=6, context_len=512)
cfg_m = ModelConfig.modern(vocab_size=8192, d_model=320, n_heads=8, n_layers=6, context_len=512)
mc, mm = ClassicalGPT(cfg_c), ClassicalGPT(cfg_m)
print(f"Classical: {mc.param_breakdown()['total']:,} params")
print(f"Modern   : {mm.param_breakdown()['total']:,} params")
pd.DataFrame({
    "field": ["norm","pos","mlp","bias"],
    "classical": [cfg_c.norm, cfg_c.pos, cfg_c.mlp, cfg_c.bias],
    "modern": [cfg_m.norm, cfg_m.pos, cfg_m.mlp, cfg_m.bias],
})

Classical: 10,183,680 params
Modern   : 10,059,840 params


,field,classical,modern
0,norm,layernorm,rmsnorm
1,pos,learned,rope
2,mlp,gelu,swiglu
3,bias,True,False


In [3]:
# モジュール木を表示（手書き実装が読めることを確認）
def tree(module, prefix="", depth=0, maxd=2):
    for name, child in module.named_children():
        n = sum(p.numel() for p in child.parameters())
        print(f"{'  '*depth}{name}: {child.__class__.__name__}  ({n:,} params)")
        if depth < maxd:
            tree(child, prefix, depth+1, maxd)
tree(mc, maxd=1)
print("--- block 0 ---")
tree(mc.blocks[0], maxd=1)

tok_emb: Embedding  (2,621,440 params)
pos_emb: Embedding  (163,840 params)
drop: Dropout  (0 params)
blocks: ModuleList  (7,397,760 params)
  0: TransformerBlock  (1,232,960 params)
  1: TransformerBlock  (1,232,960 params)
  2: TransformerBlock  (1,232,960 params)
  3: TransformerBlock  (1,232,960 params)
  4: TransformerBlock  (1,232,960 params)
  5: TransformerBlock  (1,232,960 params)
ln_f: LayerNorm  (640 params)
lm_head: Linear  (2,621,440 params)
--- block 0 ---
ln1: LayerNorm  (640 params)
attn: CausalSelfAttention  (410,880 params)
  qkv: Linear  (308,160 params)
  proj: Linear  (102,720 params)
  resid_dropout: Dropout  (0 params)
ln2: LayerNorm  (640 params)
mlp: MLP  (820,800 params)
  fc: Linear  (410,880 params)
  act: GELU  (0 params)
  proj: Linear  (409,920 params)
  dropout: Dropout  (0 params)


## Observation / Interpretation / Caveat
- **Observation**: Classical と Modern はパラメータ数がほぼ同じ（SwiGLU の隠れ幅を 8d/3 に調整、RoPE は位置パラメータ0）。差はアーキテクチャのみ。
- **Interpretation**: これにより M3 のアブレーションが「容量」ではなく「構造」の効果を測れる（ceteris paribus）。
- **Caveat**: パラメータ一致は近似（±2%以内）。FLOPs はわずかに異なる（NB08）。

**次へ**: [08_parameter_and_flops_analysis](08_parameter_and_flops_analysis.ipynb)